# 12 · Evaluation, Backtesting & Decision-Making: putting it all together

This notebook builds a complete **probabilistic-forecast evaluation pipeline** for a monthly series, then turns
forecasts into **decisions**.

## Theory

### 1. Evaluation protocol checklist
1. **Rolling-origin backtest**: many origins, refit (or re-filter) at each one, forecast $h=1..H$.
2. **Benchmark**: seasonal naive with bootstrapped errors. Report **skill scores** $1-S/S_{\text{bench}}$.
3. **Proper scores**: CRPS (or log score) as the headline. Pinball at the quantiles you actually use.
4. **Calibration diagnostics**: PIT histograms and interval coverage, **per horizon**.
5. **Sharpness**: interval widths. Compare only among calibrated forecasters.
6. **Significance**: score differences are autocorrelated (overlapping horizons), so use a **Diebold–Mariano**
   test with HAC variance rather than eyeballing averages.
7. **Many series**: use scale-free versions, e.g. CRPS or pinball divided by the in-sample MAE of the seasonal naive
   (like MASE, or M5's *scaled pinball loss*), or weighted-QL.

### 2. Which score for which forecast type
| Forecast type | Primary score | Diagnostics |
|---|---|---|
| Full distribution / samples | CRPS, log score | PIT, rank histogram |
| Quantiles | pinball, wQL | quantile coverage, crossing |
| Intervals | interval (Winkler) score | coverage, width |
| Event probabilities | Brier, log loss | reliability diagram, resolution, value curve |
| Categories / bins | RPS, ignorance | reliability per class |
| Multivariate / hierarchy | energy, variogram, CRPS per level | aggregate coverage, coherence |

### 3. Decision theory: why quantiles are the right output
**Newsvendor problem**: you order $Q$ before demand $Y$ is known. Unsold stock costs $c_o$ per unit (overage),
unmet demand costs $c_u$ (underage, lost margin). The expected cost
$\mathbb E[c_u(Y-Q)^+ + c_o(Q-Y)^+]$ is minimised at
$$Q^*=F^{-1}\!\left(\tau^*\right),\qquad\tau^*=\frac{c_u}{c_u+c_o}\quad(\text{the critical ratio}).$$
The cost is *exactly* a scaled pinball loss at $\tau^*$. So **ordering the mean or median is sub-optimal whenever $c_u\ne c_o$**.

Related ideas: **safety stock** $=Q^*-\text{median}$; **cycle service level** $=\Pr(Y\le Q)=\tau$;
fill rate (expected share of demand served) is a different metric that needs the whole distribution.

### 4. Common pitfalls
* Evaluating on one origin. Using a confidence interval as a prediction interval. Looking only at coverage.
* Summing quantiles across series or months. Ignoring dependence in aggregates.
* Leakage: calibrating on the test period, tuning on the backtest, then reporting the same backtest.
* Comparing log scores across different transformations without the Jacobian.
* Mean of a log-model forecast without bias adjustment ($e^{\mu+\sigma^2/2}$).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(12)

y = pf.load_retail_total()
ly = np.log(y)
H, M, FIRST = 6, 500, 60
TAUS = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

## 1. Rolling backtest of four forecasters (all represented as 500 samples)
* **Benchmark**: seasonal naive + drift with bootstrapped log seasonal differences
* **SARIMA**: airline model on log y (log-normal predictive)
* **ETS(M,A,M)**: simulated paths
* **Pool**: equal-weight linear pool of SARIMA and ETS (nb 05)

In [2]:
def snaive_boot(series, H, M):
    z = np.log(series.to_numpy()); d = z[12:] - z[:-12]; pool = d - d.mean()
    zz = np.tile(np.r_[z, np.zeros(H)], (M, 1)); n = len(z)
    for t in range(n, n + H):
        zz[:, t] = zz[:, t - 12] + d.mean() + rng.choice(pool, M)
    return np.exp(zz[:, n:])

records = []           # one row per (origin, h, model)
samples = {}           # (origin, model) -> (M, h) samples
for o in range(FIRST, len(y) - 1):
    tr = y.iloc[:o]
    hmax = min(H, len(y) - o)
    obs = y.iloc[o:o + hmax].to_numpy()
    res = SARIMAX(np.log(tr), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    fc = res.get_forecast(hmax)
    m, s = np.asarray(fc.predicted_mean), np.sqrt(np.asarray(fc.var_pred_mean))
    ets = ETSModel(tr, error="mul", trend="add", seasonal="mul", seasonal_periods=12).fit(disp=False)
    sm_ = {"Benchmark (snaive boot)": snaive_boot(tr, hmax, M),
           "SARIMA": np.exp(m + s * rng.standard_normal((M, hmax))),
           "ETS(M,A,M)": np.asarray(ets.simulate(hmax, repetitions=M, anchor="end")).reshape(hmax, M).T}
    sm_["Pool (SARIMA+ETS)"] = np.vstack([sm_["SARIMA"][: M // 2], sm_["ETS(M,A,M)"][: M // 2]])
    scale = np.mean(np.abs(tr.to_numpy()[12:] - tr.to_numpy()[:-12]))      # in-sample seasonal-naive MAE
    for name, smp in sm_.items():
        samples[(o, name)] = smp
        q = np.quantile(smp, TAUS, axis=0)
        for h in range(hmax):
            r = dict(origin=o, date=y.index[o + h], h=h + 1, model=name, y=obs[h], scale=scale,
                     crps=float(pf.crps_sample(obs[h], smp[:, h])), pit=float(np.mean(smp[:, h] <= obs[h])),
                     median=q[3, h], mean=smp[:, h].mean())
            for k, tau in enumerate(TAUS):
                r[f"q{tau}"] = q[k, h]
            records.append(r)
bt = pd.DataFrame(records)
print(f"{bt.origin.nunique()} origins x up to {H} horizons x {bt.model.nunique()} models = {len(bt)} forecasts")

59 origins x up to 6 horizons x 4 models = 1376 forecasts


## 2. Scoreboard

In [3]:
def summarize(g):
    return pd.Series({
        "CRPS": g.crps.mean(),
        "scaled CRPS": (g.crps / g.scale).mean(),
        "pinball@0.9": pf.pinball_loss(g.y, g["q0.9"], 0.9).mean(),
        "wQL": pf.weighted_quantile_loss(g.y.to_numpy(), {t: g[f"q{t}"].to_numpy() for t in TAUS}),
        "cov 80%": pf.coverage(g.y, g["q0.1"], g["q0.9"]),
        "cov 90%": pf.coverage(g.y, g["q0.05"], g["q0.95"]),
        "IS 90%": pf.interval_score(g.y, g["q0.05"], g["q0.95"], 0.1).mean(),
        "width 90%": (g["q0.95"] - g["q0.05"]).mean(),
        "MAE (median)": np.abs(g.y - g["median"]).mean(),
    })
board = bt.groupby("model").apply(summarize)
board.insert(1, "CRPS skill vs bench", 1 - board.CRPS / board.loc["Benchmark (snaive boot)", "CRPS"])
board.sort_values("CRPS")

,CRPS,CRPS skill vs bench,scaled CRPS,pinball@0.9,wQL,cov 80%,cov 90%,IS 90%,width 90%,MAE (median)
model,,,,,,,,,,
Pool (SARIMA+ETS),79.741,0.086,0.639,25.688,0.033,0.802,0.901,601.573,505.089,110.355
"ETS(M,A,M)",79.853,0.085,0.640,25.856,0.033,0.770,0.863,614.721,462.895,110.737
SARIMA,83.735,0.040,0.671,26.104,0.035,0.802,0.922,606.394,520.795,117.613
Benchmark (snaive boot),87.227,0.000,0.701,30.918,0.037,0.808,0.913,668.482,545.070,125.979


In [4]:
MODELS = list(bt.model.unique())                       # insertion order: benchmark, SARIMA, ETS, pool
COLOR = dict(zip(MODELS, pf.SERIES))
cov90 = bt.assign(cov=lambda d: (d.y >= d["q0.05"]) & (d.y <= d["q0.95"])).groupby(["h", "model"])["cov"].mean().unstack()
fig = pf.subplots(1, 2, titles=["CRPS by horizon (lower is better)", "90% coverage by horizon (dashed = nominal)"], height=420)
pf.add_lines_by(fig, bt.groupby(["h", "model"]).crps.mean().unstack()[MODELS], COLOR, row=1, col=1, fmt=",.1f")
pf.add_lines_by(fig, cov90[MODELS], COLOR, row=1, col=2, showlegend=False, fmt=".2f")
fig.add_hline(y=0.9, line=dict(color=pf.INK["primary"], dash="dash", width=1), row=1, col=2)
fig.update_yaxes(range=[0.5, 1.02], row=1, col=2)
fig.update_xaxes(title_text="h (months)")
fig.update_layout(hovermode="x unified")
fig.show()

fig = pf.subplots(1, len(MODELS), titles=MODELS, shared_yaxes=True, height=340, title="PIT histograms over the backtest")
for c, mname in enumerate(MODELS, start=1):
    pf.add_pit(fig, bt[bt.model == mname].pit, bins=10, row=1, col=c, color=COLOR[mname])
fig.update_layout(margin=dict(t=100))
fig.show()

## 3. Is the difference significant? Diebold–Mariano test on CRPS
$d_t=S^{A}_t-S^{B}_t$. Under $H_0$ (equal expected score) $\text{DM}=\bar d/\sqrt{\widehat{\text{Var}}(\bar d)}\approx N(0,1)$.
The variance uses autocovariances up to lag $h-1$, because $h$-step errors from consecutive origins overlap.

In [5]:
def diebold_mariano(d, h):
    d = np.asarray(d, float); T = len(d); dbar = d.mean()
    gam = [np.sum((d[k:] - dbar) * (d[:T - k] - dbar)) / T for k in range(h)]
    var = (gam[0] + 2 * sum(gam[1:])) / T
    stat = dbar / np.sqrt(var)
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))

piv = bt.pivot_table(index=["origin", "h"], columns="model", values="crps")
rows = []
for a, b in [("SARIMA", "Benchmark (snaive boot)"), ("SARIMA", "ETS(M,A,M)"), ("Pool (SARIMA+ETS)", "SARIMA")]:
    for h in [1, 3, 6]:
        d = piv.xs(h, level="h")[a] - piv.xs(h, level="h")[b]
        stat, p = diebold_mariano(d, h)
        rows.append({"A": a, "B": b, "h": h, "mean CRPS diff (A−B)": d.mean(), "DM stat": stat, "p-value": p})
pd.DataFrame(rows)

,A,B,h,mean CRPS diff (A−B),DM stat,p-value
0,SARIMA,Benchmark (snaive boot),1,-18.200,-2.242,2.497e-02
1,SARIMA,Benchmark (snaive boot),3,-3.154,-0.421,6.735e-01
2,SARIMA,Benchmark (snaive boot),6,4.372,0.515,6.063e-01
3,SARIMA,"ETS(M,A,M)",1,-3.426,-0.903,3.663e-01
4,SARIMA,"ETS(M,A,M)",3,6.277,1.545,1.223e-01
5,SARIMA,"ETS(M,A,M)",6,7.077,4.366,1.268e-05
6,Pool (SARIMA+ETS),SARIMA,1,-0.546,-0.332,7.398e-01
7,Pool (SARIMA+ETS),SARIMA,3,-4.842,-1.886,5.924e-02
8,Pool (SARIMA+ETS),SARIMA,6,-6.332,-3.903,9.507e-05


Negative differences favour A. With about 60 origins, only fairly large differences are significant.
Be careful about declaring a winner. Rankings can also **flip with the horizon**: a model that uses the latest
observations well can win at $h=1$ and lose at $h=6$. Choose (or pool) models per horizon if that matters for the decision.

## 4. From forecasts to decisions: the newsvendor
Each month we must stock the store network one month ahead ($h=1$). Margin lost per unit short: $c_u$. Cost per
unsold unit: $c_o$. We compare order rules over the backtest:

In [6]:
h1 = bt[bt.h == 1]
def realised_cost(y_, Q, cu, co):
    return np.mean(cu * np.maximum(y_ - Q, 0) + co * np.maximum(Q - y_, 0))

cu, co = 3.0, 1.0
tau_star = cu / (cu + co)
print(f"critical ratio tau* = {tau_star:.2f}")
rows = {}
for mname in ["SARIMA", "Benchmark (snaive boot)"]:
    g = h1[h1.model == mname]
    smp = np.vstack([samples[(o, mname)][:, 0] for o in g.origin]).T          # (M, n_origins)
    Q_star = np.quantile(smp, tau_star, axis=0)
    rows[f"{mname}: order mean"] = realised_cost(g.y, g["mean"], cu, co)
    rows[f"{mname}: order median"] = realised_cost(g.y, g["median"], cu, co)
    rows[f"{mname}: order q(τ*)"] = realised_cost(g.y, Q_star, cu, co)
    rows[f"{mname}: q(τ*) service level achieved"] = np.mean(g.y.to_numpy() <= Q_star)
pd.Series(rows).to_frame("average cost per month (or service level)")

critical ratio tau* = 0.75


,average cost per month (or service level)
SARIMA: order mean,199.018
SARIMA: order median,202.430
SARIMA: order q(τ*),162.729
SARIMA: q(τ*) service level achieved,0.746
Benchmark (snaive boot): order mean,225.544
Benchmark (snaive boot): order median,227.852
Benchmark (snaive boot): order q(τ*),213.326
Benchmark (snaive boot): q(τ*) service level achieved,0.814


In [7]:
ratios = [0.5, 1, 2, 3, 5, 9, 19]
g = h1[h1.model == "SARIMA"]
smp = np.vstack([samples[(o, "SARIMA")][:, 0] for o in g.origin]).T
out = []
for r in ratios:                        # r = cu / co with co = 1
    t = r / (r + 1)
    out.append({"cu/co": r, "tau*": t, "cost: order median": realised_cost(g.y, g["median"], r, 1),
                "cost: order q(τ*)": realised_cost(g.y, np.quantile(smp, t, axis=0), r, 1)})
out = pd.DataFrame(out).set_index("cu/co")
out["saving %"] = 100 * (1 - out["cost: order q(τ*)"] / out["cost: order median"])
out

,tau*,cost: order median,cost: order q(τ*),saving %
cu/co,,,,
0.5,0.333,73.269,65.069,11.192
1.0,0.500,99.101,99.101,0.000
2.0,0.667,150.765,137.441,8.838
3.0,0.750,202.430,162.729,19.612
5.0,0.833,305.758,191.840,37.258
9.0,0.900,512.415,214.691,58.102
19.0,0.950,1029.056,252.397,75.473


The more asymmetric the costs, the more a **probabilistic** forecast saves compared with a point forecast. At
$c_u=c_o$ the optimal quantile *is* the median, so the saving is ~0.

## 5. Summary: a decision tree for choosing the forecast type
```
What does the decision need?
├── a single action with asymmetric costs (stock, capacity)  → quantile at τ* = cu/(cu+co)       (nb 02, 12)
├── a range to communicate / plan within                      → prediction interval (+ conformal) (nb 03, 04)
├── yes/no action on a threshold (alert, extra shipment)      → event probability                 (nb 06)
├── ordinal outlook for stakeholders                          → categorical / tercile forecast   (nb 07)
├── anything non-linear in the future path (totals, peaks)    → sample paths / ensembles          (nb 05)
├── several related series (stores, assets)                   → joint forecast / copula           (nb 09)
├── an aggregation hierarchy                                  → coherent reconciled forecasts     (nb 10)
└── short history / need parameter uncertainty / pooling      → Bayesian                          (nb 11)
Underneath all of them: a well-calibrated predictive distribution (nb 01, 08).
```

### Exercises
1. Add the conformal SARIMA intervals from nb 04 to the scoreboard (as quantiles at 0.05/0.95).
2. Optimise the pool weight on the first half of origins and evaluate on the second half.
3. Repeat the newsvendor analysis for each store (nb 09) and for the total, ordering from (a) the store quantiles and
   (b) the total quantile with a central warehouse. Quantify the **risk-pooling** benefit.